In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import xgboost as xgb

RANDOM_STATE = 42

# public = train, private = test (given like this in the challenge)
train_trips = pd.read_csv('public_trip_data.csv')
train_events = pd.read_csv('public_trip_event_log.csv')
train_attrs = pd.read_csv('public_trip_event_attributes.csv')

test_trips = pd.read_csv('private_trip_data.csv')
test_events = pd.read_csv('private_trip_event_log.csv')
test_attrs = pd.read_csv('private_trip_event_attributes.csv')

print('Train trips:', train_trips.shape)
print('Test trips:', test_trips.shape)

Train trips: (65289, 20)
Test trips: (21764, 13)


# EDA

In [3]:
train_trips.info()  # check dtypes + nulls first

<class 'pandas.DataFrame'>
RangeIndex: 65289 entries, 0 to 65288
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   TripID                    65289 non-null  int64  
 1   DepartureLocationCountry  65289 non-null  str    
 2   DepartureLocationCity     65289 non-null  str    
 3   ArrivalLocationCountry    65289 non-null  str    
 4   ArrivalLocationCity       65289 non-null  str    
 5   ShippingType              65289 non-null  int64  
 6   ShippingTypeDescription   65289 non-null  str    
 7   Purpose                   65289 non-null  str    
 8   OutOfPolicy               65289 non-null  str    
 9   EntitiyCode               65289 non-null  int64  
 10  EmployeeNumber            65289 non-null  str    
 11  BusinessUnit              65289 non-null  str    
 12  HotelNights               65289 non-null  int64  
 13  NetCosts                  65289 non-null  int64  
 14  Departure_CO2e   

In [4]:
# top columns with missing values
train_trips.isna().sum().sort_values(ascending=False).head(10)

TripID                      0
DepartureLocationCountry    0
DepartureLocationCity       0
ArrivalLocationCountry      0
ArrivalLocationCity         0
ShippingType                0
ShippingTypeDescription     0
Purpose                     0
OutOfPolicy                 0
EntitiyCode                 0
dtype: int64

In [5]:
# how balanced is the target
train_trips['HighCarbon'].value_counts(normalize=True)

HighCarbon
0    0.749943
1    0.250057
Name: proportion, dtype: float64

In [6]:
train_trips[['HotelNights', 'NetCosts']].describe()

,HotelNights,NetCosts
count,65289.000000,65289.000000
mean,2.936237,2151.651886
std,1.426505,1049.117452
min,1.000000,300.000000
25%,2.000000,1200.000000
50%,3.000000,2400.000000
75%,4.000000,3000.000000
max,6.000000,3900.000000


In [19]:
# main categorical cols
for col in ['Purpose', 'BusinessUnit', 'OutOfPolicy', 'ShippingTypeDescription']:
    print(col)
    print(train_trips[col].value_counts().head(5))
    print()

Purpose
Purpose
Customer Visit            28229
Conference/Exhibition     23122
Internal Business Trip     7416
Enablement                 4630
Partner Visit              1892
Name: count, dtype: int64

BusinessUnit
BusinessUnit
Sales                   21890
Marketing               12238
Value Engineering        6482
Customer Support         5336
Executive Management     4633
Name: count, dtype: int64

OutOfPolicy
OutOfPolicy
No     53284
Yes    12005
Name: count, dtype: int64

ShippingTypeDescription
ShippingTypeDescription
Economy Flight            27295
Business Class Flight     15035
BMW 3 diesel               6648
Volkswagen Golf diesel     5452
Volkswagen Golf petrol     4469
Name: count, dtype: int64



In [18]:

train_trips.groupby('HighCarbon')[['HotelNights', 'NetCosts']].mean()

,HotelNights,NetCosts
HighCarbon,,
0,2.897780,2154.990503
1,3.051574,2141.639103


In [9]:
# events log is separate from trips, check it lines up on TripID
print(train_events['TripID'].nunique(), 'trips have events out of', train_trips['TripID'].nunique())
train_events['EventName'].value_counts().head(10)

65289 trips have events out of 65289


EventName
Book Mode of Transportation    68134
Submit Travel Request          65289
Travel Request Approved        62982
Receive Confirmation           62982
Submit Expense Request         62903
Expense Request Approved       62847
Expense Reimbursement          62847
Book Lodging                   59815
Take Departure Flight          41735
Take Return Flight             41735
Name: count, dtype: int64

# Feature Engineering

In [17]:
def engineer_event_features(events):
    events = events.copy()
    events['EventTimestamp'] = pd.to_datetime(events['EventTimestamp'])

    num_events = events.groupby('TripID').size().rename('NumEvents')

    # timestamps for the events we actually care about for timing features
    key_names = ['Submit Travel Request', 'Travel Request Approved',
                 'Take Departure Flight', 'Take Departure Train', 'Pickup Rental']
    key = events[events['EventName'].isin(key_names)]
    piv = key.pivot_table(index='TripID', columns='EventName', values='EventTimestamp', aggfunc='first')
    for col in key_names:
        if col not in piv.columns:
            piv[col] = pd.NaT

    # trip could depart by flight, train, or rental 
    piv['DepartureTime'] = piv[['Take Departure Flight', 'Take Departure Train', 'Pickup Rental']].min(axis=1)
    piv['LeadTimeDays'] = (piv['DepartureTime'] - piv['Submit Travel Request']).dt.total_seconds() / 86400
    piv['ApprovalLeadDays'] = (piv['Travel Request Approved'] - piv['Submit Travel Request']).dt.total_seconds() / 86400

    # just flag whether each of these ever happened on the trip
    flag_events = {
        'HasManagerPreapproved': 'Manager Preapproved',
        'HasTripExtension': 'Trip Extension',
        'HasFlightChange': 'Flight Change',
        'HasFlightCancellation': 'Flight Cancellation',
        'HasFlightDelay': 'Flight Delay',
        'HasMissedFlight': 'Missed Flight',
        'HasHotelChange': 'Hotel Change',
        'HasModeChange': 'Mode of Transportation Change',
        'HasItineraryEdit': 'Itinerary Edit',
    }
    flags = pd.DataFrame(index=piv.index)
    for colname, evname in flag_events.items():
        trips_with_event = set(events.loc[events['EventName'] == evname, 'TripID'])
        flags[colname] = piv.index.isin(trips_with_event).astype(int)

    out = piv[['LeadTimeDays', 'ApprovalLeadDays']].join(flags).join(num_events)
    out['LeadTimeMissing'] = out['LeadTimeDays'].isna().astype(int)  # some trips never had a request logged
    return out.reset_index()


def engineer_attribute_features(attrs):
    keep = attrs[['TripID', 'DaysPreapproved', 'ExtensionLength']].copy()
    keep['DaysPreapproved'] = keep['DaysPreapproved'].fillna(0)
    keep['ExtensionLength'] = keep['ExtensionLength'].fillna(0)
    return keep


train_ev_feats = engineer_event_features(train_events)
train_at_feats = engineer_attribute_features(train_attrs)
train_df = train_trips.merge(train_ev_feats, on='TripID', how='left').merge(train_at_feats, on='TripID', how='left')

test_ev_feats = engineer_event_features(test_events)
test_at_feats = engineer_attribute_features(test_attrs)
test_df = test_trips.merge(test_ev_feats, on='TripID', how='left').merge(test_at_feats, on='TripID', how='left')

print(train_df.shape, test_df.shape)
train_df.head(3)

(65289, 35) (21764, 28)


,TripID,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingType,ShippingTypeDescription,Purpose,OutOfPolicy,EntitiyCode,...,HasFlightCancellation,HasFlightDelay,HasMissedFlight,HasHotelChange,HasModeChange,HasItineraryEdit,NumEvents,LeadTimeMissing,DaysPreapproved,ExtensionLength
0,1,CN,Beijing,IN,New Delhi,12,Business Class Flight,Customer Visit,No,9000,...,0,0,0,0,0,0,10,0,2,0.0
1,3,US,New York,MX,Mexico City,11,First Class Flight,Customer Visit,Yes,6000,...,0,0,0,0,0,0,10,0,2,0.0
2,6,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Conference/Exhibition,No,9000,...,0,0,0,0,0,0,10,0,1,0.0


In [11]:
# these are all derived from / literally the target, so they can't be features
LEAKAGE_COLS = ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e', 'HighCarbon']
ID_COLS = ['TripID', 'EmployeeNumber']

CATEGORICAL_COLS = [
    'DepartureLocationCountry', 'DepartureLocationCity', 'ArrivalLocationCountry', 'ArrivalLocationCity',
    'ShippingTypeDescription', 'Purpose', 'OutOfPolicy', 'BusinessUnit'
]
NUMERIC_COLS = [
    'ShippingType', 'EntitiyCode', 'HotelNights', 'NetCosts',
    'LeadTimeDays', 'ApprovalLeadDays', 'NumEvents', 'LeadTimeMissing',
    'HasManagerPreapproved', 'HasTripExtension', 'HasFlightChange', 'HasFlightCancellation',
    'HasFlightDelay', 'HasMissedFlight', 'HasHotelChange', 'HasModeChange', 'HasItineraryEdit',
    'DaysPreapproved', 'ExtensionLength'
]

print(f'Using {len(CATEGORICAL_COLS)} categorical features and {len(NUMERIC_COLS)} numeric features.')
print(f'Explicitly excluded (leakage risk): {LEAKAGE_COLS}')

Using 8 categorical features and 19 numeric features.
Explicitly excluded (leakage risk): ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e', 'HighCarbon']


# Train-validation split

In [12]:
y = train_df['HighCarbon']

# Keep the raw feature tables separate. We split BEFORE fitting any preprocessing.
X_raw = train_df[CATEGORICAL_COLS + NUMERIC_COLS].copy()
X_test_raw = test_df[CATEGORICAL_COLS + NUMERIC_COLS].copy()

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Preprocessing is fitted ONLY on X_train, then applied to validation/test.
# This prevents validation information from leaking into the preprocessing step.
X_train = X_train_raw.copy()
X_val = X_val_raw.copy()
X_test = X_test_raw.copy()

for col in ['LeadTimeDays', 'ApprovalLeadDays']:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[CATEGORICAL_COLS] = encoder.fit_transform(X_train[CATEGORICAL_COLS])
X_val[CATEGORICAL_COLS] = encoder.transform(X_val[CATEGORICAL_COLS])
X_test[CATEGORICAL_COLS] = encoder.transform(X_test[CATEGORICAL_COLS])

print(X_train.shape, X_val.shape, X_test.shape)
X_train.head(3)


(52231, 27) (13058, 27) (21764, 27)


,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingTypeDescription,Purpose,OutOfPolicy,BusinessUnit,ShippingType,EntitiyCode,...,HasTripExtension,HasFlightChange,HasFlightCancellation,HasFlightDelay,HasMissedFlight,HasHotelChange,HasModeChange,HasItineraryEdit,DaysPreapproved,ExtensionLength
47891,3.0,1.0,5.0,12.0,7.0,3.0,0.0,0.0,14,8000,...,0,0,0,0,0,0,0,0,3,0.0
21565,0.0,8.0,16.0,22.0,3.0,0.0,0.0,0.0,10,6000,...,0,0,0,0,0,0,1,0,2,0.0
5156,0.0,8.0,18.0,11.0,2.0,4.0,1.0,10.0,12,9000,...,0,0,0,0,0,0,0,0,0,0.0


In [13]:
model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)  # threshold at 0.5 for accuracy

auc = roc_auc_score(y_val, val_probs)
acc = accuracy_score(y_val, val_preds)
print(f'Validation ROC-AUC: {auc:.4f}')
print(f'Validation Accuracy: {acc:.4f}')
print()
print(classification_report(y_val, val_preds))


Validation ROC-AUC: 0.9994
Validation Accuracy: 0.9936

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      9793
           1       0.99      0.98      0.99      3265

    accuracy                           0.99     13058
   macro avg       0.99      0.99      0.99     13058
weighted avg       0.99      0.99      0.99     13058



In [14]:
# what's actually driving the model
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances.head(10)

DepartureLocationCountry    0.219618
ArrivalLocationCountry      0.170880
ShippingTypeDescription     0.153834
ArrivalLocationCity         0.149795
DepartureLocationCity       0.089949
ShippingType                0.087316
OutOfPolicy                 0.041265
HotelNights                 0.038432
LeadTimeDays                0.003213
HasMissedFlight             0.003101
dtype: float32

In [15]:
# Now that model parameters look good on validation, refit preprocessing on ALL public training data.
X = X_raw.copy()
X_test = X_test_raw.copy()

for col in ['LeadTimeDays', 'ApprovalLeadDays']:
    median_val = X[col].median()
    X[col] = X[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

final_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[CATEGORICAL_COLS] = final_encoder.fit_transform(X[CATEGORICAL_COLS])
X_test[CATEGORICAL_COLS] = final_encoder.transform(X_test[CATEGORICAL_COLS])

final_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=RANDOM_STATE
)
final_model.fit(X, y)

test_probs = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TripID': test_df['TripID'],
    'HighCarbon': test_probs
})

print(submission.shape)
print(submission['TripID'].is_unique)
submission.head()


(21764, 2)
True


,TripID,HighCarbon
0,2,0.999845
1,4,0.006202
2,5,0.899262
3,14,0.000189
4,15,0.000251


# Submission file

In [16]:
submission.to_csv('predictions.csv', index=False)
print('Saved predictions.csv')
submission['HighCarbon'].describe()

Saved predictions.csv


count    21764.000000
mean         0.248453
std          0.423647
min          0.000002
25%          0.000177
50%          0.000560
75%          0.142836
max          0.999979
Name: HighCarbon, dtype: float64

In [21]:
predictions = pd.read_csv('predictions.csv')
predictions.head()

,TripID,HighCarbon
0,2,0.999845
1,4,0.006202
2,5,0.899262
3,14,0.000189
4,15,0.000251


## Summary

Trained an XGBoost classifier to predict HighCarbon. Added EDA at the start to check missing values and class balance before doing anything else, then built lead-time / approval-time / event-flag features from the event log and joined everything onto the trip table. Dropped the CO2e columns and employee number since those either leak the label directly or aren't useful for generalizing. Validation ROC-AUC is in the train/val split cell above, feature importances right after. Final model is refit on the full training set and predictions.csv has probabilities for the private test set.